# Le pretrain MAE lisse-t-il les manoeuvres ?

Un MAE apprend à reconstruire une dynamique orbitale lisse en interpolant les patchs masqués.
Une manoeuvre est justement l'endroit où la dynamique casse. Si la pression de reconstruction
pousse l'encodeur à lisser les discontinuités, il jette l'information qu'on veut détecter — et
aucun finetuning ne la retrouvera.

Le test : masquer **exactement le patch qui contient une manoeuvre connue** (labels DORIS) et
regarder ce que le décodeur remet à la place. Deux issues possibles :

- il restitue le saut → le backbone a appris que ces ruptures existent, le pretrain est utile ;
- il remet une interpolation lisse → il traite la manoeuvre comme du bruit à effacer.

Le résidu de reconstruction sur les patchs à manoeuvre, comparé aux patchs sans manoeuvre,
quantifie la réponse.

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

from ml.inference import load_checkpoint, reconstruct_window, patch_of_index
from ml.datahandler import load_doris_objects, build_features

base = Path.cwd()
device = torch.device('cpu')

CKPT_ID = '2026-08-11_14-43-36'  ## <-- le run de pretrain à diagnostiquer
ckpt_path = Path(os.path.join(base, '..', '..', 'outputs', 'ml', 'pretrain', CKPT_ID, 'checkpoints', 'best.pt'))
doris_dir = Path(os.path.join(base, '..', '..', 'data', 'parsed', 'labelled_leo_DORIS'))

mae, cfg, mean, scale = load_checkpoint(ckpt_path, device)
window_size, patch_size = cfg.data.window_size, cfg.model.patch_size
print(f"MAE {cfg.model.name} | window {window_size} | patch {patch_size} | "
      f"{mae.num_patches} patchs | masking_ratio {mae.masking_ratio}")

In [ ]:
## Données DORIS, normalisées avec le scaler DU PRETRAIN : le MAE n'a de sens que sur
## la distribution d'entrée sur laquelle il a été entraîné.
objects, labels = load_doris_objects(doris_dir)

per_obj, feature_cols = {}, None
for norad, df in objects.items():
    features, feature_cols = build_features(df, spacetrack=True)
    per_obj[norad] = (features[feature_cols].to_numpy(np.float32) - mean) / scale

## un TimeIndex par manoeuvre projetable, cf. load_doris_objects
maneuvers = {norad: labels[(labels['norad_id'] == norad) & labels['TimeIndex'].notna()]['TimeIndex']
                        .astype(int).to_numpy()
             for norad in per_obj}

print(feature_cols)
for norad, X in per_obj.items():
    print(f"  norad {norad}: L={len(X)}  manoeuvres projetables={len(maneuvers[norad])}")

In [ ]:
def window_around(norad, time_index):
    """Fenêtre de window_size centrée sur time_index, recalée si on touche un bord.
    Retourne (x normalisé (F,W), window_start, patch contenant la manoeuvre)."""
    X = per_obj[norad]
    start = int(np.clip(time_index - window_size // 2, 0, len(X) - window_size))
    x = X[start:start + window_size].T  ## (F, W)
    return x, start, patch_of_index(time_index, start, patch_size)


def denormalize(x):
    """(F,W) normalisé -> unités physiques (sma en km, etc.)"""
    return x.T * scale + mean


## un cas concret : la première manoeuvre du premier objet qui en a une
norad = next(n for n, m in maneuvers.items() if len(m))
time_index = int(maneuvers[norad][0])
x, start, patch = window_around(norad, time_index)

recon, masked = reconstruct_window(mae, x, masked_patches=[patch], device=device)
print(f"norad {norad} | manoeuvre au TimeIndex {time_index} | fenêtre [{start}, {start + window_size}) "
      f"| patch masqué {patch} sur {mae.num_patches}")

In [ ]:
## On ne trace pas dt : sa loss de reconstruction est à poids nul au pretrain (cf. train.py),
## le décodeur n'a jamais été entraîné à le restituer.
TO_PLOT = ['sma', 'sma_diff', 'k', 'h']

x_phys, recon_phys = denormalize(x), denormalize(recon)
t = np.arange(start, start + window_size)
patch_slice = slice(patch * patch_size, (patch + 1) * patch_size)

fig, axes = plt.subplots(len(TO_PLOT), 1, figsize=(14, 3 * len(TO_PLOT)), sharex=True)
for ax, name in zip(axes, TO_PLOT):
    col = feature_cols.index(name)
    ax.plot(t, x_phys[:, col], color='black', lw=1.4, label='original')
    ax.plot(t, recon_phys[:, col], color='tab:red', lw=1.2, ls='--', label='reconstruit')
    ax.axvspan(t[patch_slice][0], t[patch_slice][-1], color='tab:orange', alpha=0.18,
               label='patch masqué (manoeuvre)')
    ax.axvline(time_index, color='tab:blue', lw=1.0, label='epoch manoeuvre')
    ax.set_ylabel(name)
    ax.legend(loc='upper right', fontsize=8)

axes[0].set_title(f"norad {norad} — reconstruction MAE du patch contenant la manoeuvre {time_index}")
axes[-1].set_xlabel('TimeIndex (TLE)')
fig.tight_layout()
plt.show()

In [ ]:
## Quantitatif : résidu de reconstruction sur les patchs À manoeuvre vs SANS manoeuvre.
## Dans les deux cas on masque UN seul patch, donc la difficulté est comparable — seule
## la présence de la discontinuité change.
CHANNEL = 'sma_diff'  ## la signature d'une manoeuvre in-track
col = feature_cols.index(CHANNEL)
rng = np.random.default_rng(0)

def patch_rmse(x, recon, patch):
    sl = slice(patch * patch_size, (patch + 1) * patch_size)
    return float(np.sqrt(np.mean((x[col, sl] - recon[col, sl]) ** 2)))

with_maneuver, without = [], []
for norad, indices in maneuvers.items():
    if len(per_obj[norad]) < window_size:
        continue
    for time_index in indices:
        x, start, patch = window_around(norad, time_index)
        if not 0 <= patch < mae.num_patches:
            continue
        ## patchs de la même fenêtre ne contenant aucune manoeuvre : le témoin
        occupied = {patch_of_index(i, start, patch_size) for i in indices}
        free = [p for p in range(mae.num_patches) if p not in occupied]
        if not free:
            continue
        control = int(rng.choice(free))

        recon_m, _ = reconstruct_window(mae, x, masked_patches=[patch], device=device)
        recon_c, _ = reconstruct_window(mae, x, masked_patches=[control], device=device)
        with_maneuver.append(patch_rmse(x, recon_m, patch))
        without.append(patch_rmse(x, recon_c, control))

with_maneuver, without = np.array(with_maneuver), np.array(without)
print(f"{len(with_maneuver)} manoeuvres | résidu RMSE sur {CHANNEL} (unités normalisées)")
print(f"  patch AVEC manoeuvre : médiane {np.median(with_maneuver):.4f}  moyenne {with_maneuver.mean():.4f}")
print(f"  patch SANS manoeuvre : médiane {np.median(without):.4f}  moyenne {without.mean():.4f}")
print(f"  ratio des médianes   : {np.median(with_maneuver) / np.median(without):.2f}x")

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.histogram_bin_edges(np.concatenate([with_maneuver, without]), bins=40)
ax.hist(without, bins=bins, alpha=0.6, label='patch sans manoeuvre', color='tab:blue')
ax.hist(with_maneuver, bins=bins, alpha=0.6, label='patch avec manoeuvre', color='tab:red')
ax.set_xlabel(f'RMSE de reconstruction ({CHANNEL}, normalisé)')
ax.set_ylabel('nombre de patchs')
ax.set_title('Le décodeur reconstruit-il aussi bien avec et sans discontinuité ?')
ax.legend()
fig.tight_layout()
plt.show()

## Comment lire le résultat

**Ratio proche de 1** — le décodeur reconstruit les patchs à manoeuvre aussi bien que les autres :
la discontinuité est représentée dans les features de l'encodeur. Le pretrain est exploitable, et
un finetuning médiocre vient d'ailleurs (readout, cible, normalisation).

**Ratio nettement supérieur à 1** — le décodeur échoue spécifiquement là où il y a une manoeuvre,
et le graphe du haut montrera une interpolation lisse en travers du saut. Deux lectures opposées,
à trancher avec le graphe :

- si la reconstruction est lisse *et* l'encodeur ne voit pas le patch masqué, c'est normal : il ne
  peut pas deviner une manoeuvre imprévisible. Ce n'est **pas** une preuve que l'information est perdue,
  puisque au finetuning aucun patch n'est masqué et l'encodeur voit la rupture ;
- ce qui serait alarmant, c'est un résidu élevé sur les patchs *voisins* non masqués, signe que la
  représentation elle-même écrase la discontinuité.

Pour trancher, refais tourner avec `masked_patches=[]` (aucun masque) : l'encodeur voit alors toute
la fenêtre, exactement comme au finetuning. Si le saut ne ressort toujours pas de la reconstruction,
l'information est effectivement absente des features et le pretrain est à revoir.